# Baselines — the numbers every fine-tune must beat (Kaggle GPU)

Runs each zero-shot baseline over every eval slice, writes predictions, and
regenerates the baseline report. Thin driver: one prediction path
(`run_over_manifest`) for baselines, our model, and the commercial APIs alike.

Prerequisite: run `prepare_data.ipynb` first so `data/*.manifest.jsonl` exist.

In [ ]:
!pip install -q 'transformers>=4.44' librosa soundfile
!pip install -q -e /kaggle/working/shono

In [ ]:
from pathlib import Path

from shono.api import run_over_manifest
from shono.data import Manifest
from shono.provenance import RunContext, seed_everything
from shono.transcribe import ShortFormWhisperTranscriber

REPO = Path('/kaggle/working/shono')
DATA = REPO / 'data'
PREDS = REPO / 'reports' / 'predictions'
PREDS.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT = Path('/kaggle/input')   # manifest audio paths resolve under here
seed_everything(0)

# system slug -> (report system name, HF model id). Confirm the tugstugi base id.
SYSTEMS = {
    'large-v3':        ('whisper-large-v3 (zero-shot)',        'openai/whisper-large-v3'),
    'large-v3-turbo':  ('whisper-large-v3-turbo (zero-shot)',  'openai/whisper-large-v3-turbo'),
    'tugstugi-medium': ('tugstugi whisper-medium (zero-shot)', 'openai/whisper-medium'),
}
SLICES = ['cv-bn-test', 'fleurs-bn-test', 'bengali-loop-sample']

## Transcribe every slice with every baseline

Each segment is transcribed on its own window, so the same short-form transcriber
scores read and long-form slices alike. Predictions carry the run's provenance.

In [ ]:
for slug, (system_name, model_id) in SYSTEMS.items():
    transcriber = ShortFormWhisperTranscriber(model_id, language='bn')
    ctx = RunContext.capture(0, {'model': model_id}, command='baselines.ipynb', repo=str(REPO))
    for slice_name in SLICES:
        manifest = Manifest.from_jsonl(DATA / f'{slice_name}.manifest.jsonl')
        preds = run_over_manifest(
            transcriber, manifest, system_name, audio_root=AUDIO_ROOT, run_context=ctx,
        )
        out = PREDS / f'{slug}__{slice_name}.jsonl'
        preds.to_jsonl(out)
        print(f'{slug} x {slice_name}: {len(preds.hypotheses)} predictions -> {out.name}')

## Generate the baseline report

Fills the skeleton with real, CI-backed numbers. Commit `reports/baselines.{md,json}`
as the milestone evidence.

In [ ]:
!cd /kaggle/working/shono && python -m shono.eval --report baselines